## Reading 37-2 - Riding Over Transients

By the end of this reading, you will learn:
<ol>
    <li>Why a fault response should never trip on a single bad reading</li>
    <li>How a leaky bucket counter separates noise from a real fault</li>
    <li>How hysteresis keeps a controller from switching back and forth on noise</li>
    <li>How watchdogs, escalation, and event reports turn many fault responses into one coherent plan</li>
</ol>

### Noise Is Not a Fault - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec37/leaky_bucket.c">leaky_bucket.c</a>

Real sensors are noisy. A battery voltage that is truly 28 V will occasionally read 25 V for one sample, because of electrical interference or a glitch in the converter. Ama's rule from his slide on noise: <b>never trip a fault response on one spike</b>. A spacecraft that switches to its backup battery every time a reading glitches will soon have no healthy battery left to switch to.

The example samples a battery once per second. Below 26.0 V is under-voltage. The data contains three isolated spikes, a two-sample burst, and then, starting at sample 20, a real failure. The naive check declares a fault on the first bad sample. The <b>leaky bucket</b> instead keeps a count: each bad sample adds 2, each good sample drains 1, and the fault is declared only when the count reaches 6:

    if (bad) {
    	bucket += FILL_PER_BAD;
    	if (bucket > BUCKET_MAX) {
    		bucket = BUCKET_MAX;
    	}
    }
    else if (bucket >= LEAK_PER_GOOD) {
    	bucket -= LEAK_PER_GOOD;
    }

    $ make leaky_bucket
    sample    volts     bad?   bucket
         2     25.2      BAD        2
         3     28.2                 1
         4     28.1                 0
       ...
        12     25.1      BAD        2
        13     25.3      BAD        4
        14     28.0                 3
       ...
        20     25.6      BAD        2
        21     25.4      BAD        4
        22     25.2      BAD        6

    Naive check declared under-voltage at sample 2
    Leaky bucket declared under-voltage at sample 22

The bucket rode over every isolated spike and even the two-sample burst, then caught the real failure two seconds after it began. Two things about the design:
<ul>
    <li><b>The settings are a trade-off.</b> A larger trip level means fewer false alarms but a slower response to a real fault. Engineers choose the numbers from the sensor's measured noise, which is why Ama's slide says to characterize the noise before choosing a filter.</li>
    <li><b>The bucket itself is bounded.</b> Without <code>BUCKET_MAX</code>, a long fault would keep adding to the count until the <code>int</code> overflowed. Every counter needs a limit, as you learned in Lecture 36.</li>
</ul>

### Hysteresis - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec37/thermostat.c">thermostat.c</a>

Ama's table of subsystems lists one failure for thermal control: <i>thermostat logic without hysteresis</i>. A heater keeps an instrument near 20.0 C, and the temperature sensor has noise of up to 0.3 C. The obvious rule is "heater on below 20, off above 20." But when the temperature is near 20, noise pushes each reading randomly to one side or the other:

    $ make thermostat
    Without hysteresis:  503 heater switches, temperature 19.00 to 20.26 C
    With hysteresis:      53 heater switches, temperature 19.00 to 20.66 C

Over 1000 steps, the heater switched on or off 503 times, about every other step. Each switch wears out a relay and causes a spike in the power system.

<b>Hysteresis</b> uses two thresholds instead of one: turn on below 19.5, turn off above 20.5, and in between, <i>change nothing</i>:

    if (measured < SETPOINT - BAND) {
    	want_on = 1;
    }
    else if (measured > SETPOINT + BAND) {
    	want_on = 0;
    }
    else {
    	want_on = heater_on;	/* inside the band: change nothing */
    }

That cut the switching by a factor of almost 10. The cost is a wider swing in temperature, up to 20.66 C instead of 20.26 C. Choosing the band means deciding which matters more for this instrument: tight temperature or a long-lived heater.

### Watchdogs, Escalation, and Event Reports

Some faults do not show up in any reading. The software itself can hang, as the unbounded search did in Lecture 34. A <b>watchdog</b> is a hardware timer that resets the processor unless the software "pets" it regularly. The rule that makes it work: pet the watchdog only from the main loop, and only after confirming that the work actually got done. A watchdog petted from a timer interrupt will keep a hung program alive forever.

When a fault is confirmed, the response should be the smallest one that fixes it. Ama's slide on fault management describes an <b>escalation</b> ladder:
<ol>
    <li><b>Retry</b> the operation</li>
    <li><b>Reset</b> the unit</li>
    <li><b>Swap</b> to the redundant unit</li>
    <li>Enter <b>safe mode</b>: point the solar panels at the Sun, keep warm, listen for the ground, as the command queue did in Lecture 34</li>
</ol>

Each step happens only if the one before it did not work. And every subsystem's fault responses must add up to <i>one coherent response</i>: the power system shedding load at the same moment the thermal system turns on every heater would be a disaster.

Finally, every fault and every response is recorded as an <b>event report (EVR)</b>, time-tagged using the clock from Lecture 36, with a severity of INFO, WARN, ERROR, or FATAL. When something goes wrong a hundred million miles away, these records are often all the ground team has to work out what happened.

### <font color = "red">Class Introduction Question #4 - In <code>leaky_bucket.c</code>, why did the naive check trip at sample 2 while the bucket waited until sample 22? What happens to the response time if you raise <code>TRIP_LEVEL</code>?</font>

### <font color = "red">Class Introduction Question #5 - Explain why the heater without hysteresis switched 503 times. What did hysteresis cost in exchange for reducing that to 53?</font>

### <font color = "red">Class Introduction Question #6 - Why should a watchdog never be petted from a timer interrupt? Describe the four steps of the escalation ladder.</font>

### In class, we will build a fault protection chain for redundant temperature sensors: vote, ride over transients, control a heater, and report every event.